# Day 02 — preprocessing финансовых предложений

## D2-01 — контракт минимальной очистки и setup

**Цель:** определить policy `minimal-v1` и загрузить реальные исходные данные.
Вход — локальный canonical CSV Financial PhraseBank `sentences_75agree`;
на этом checkpoint выход — исходный DataFrame и согласованный контракт.
Очистка появится в D2-02, обработка таблицы — в D2-03.

**Краткая теория.** Preprocessing преобразует представление текста.
Удаление `not` меняет отрицание, удаление `-5%` теряет направление и величину
изменения. Поэтому сохраняем эти сигналы. Детерминированная очистка не требует
`fit`; TF-IDF изучает vocabulary/IDF и позже будет обучаться только на train.

Источник: takala/financial_phrasebank; Malo et al. (2014); CC-BY-NC-SA-3.0.
Если локального CSV нет, подготовьте его отдельной командой из корня:
`PYTHONPATH=src venv/bin/python -m finnews_sentiment.data.prepare_phrasebank --download`.
Notebook не скачивает данные.

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").is_file() and (candidate / "src/finnews_sentiment").is_dir():
            return candidate
    raise FileNotFoundError("Project root with SPEC.md and src/finnews_sentiment not found")


ROOT: Path = find_project_root(Path.cwd().resolve())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from finnews_sentiment.data.load_data import CLASS_ORDER, load_raw_news

print(f"Interpreter: {sys.executable}")
print(f"Project root: {ROOT}")
print(f"Class order: {CLASS_ORDER}")

Interpreter: /home/mkkr/projects/Text-classification/venv/bin/python
Project root: /home/mkkr/projects/Text-classification
Class order: ('negative', 'neutral', 'positive')


### Вызов кода: загрузка canonical CSV

Проверяем файл до/после чтения и совпадение повторной загрузки.
Это проверка сохранности raw representation, а не применение preprocessing.
На checkpoint D2-01 `features.preprocess` не импортировался: тогда это была
непринятая заготовка с NLTK resource access при import. Ниже, в D2-02,
импортируется уже проверенная реализация очистки одного текста.

In [2]:
import hashlib
import pandas as pd
from IPython.display import display

CANONICAL_PATH = ROOT / "data/interim/financial_phrasebank_75agree.csv"
if not CANONICAL_PATH.is_file():
    raise FileNotFoundError("Canonical CSV is missing; run the explicit preparation command above")

canonical_sha256_before = hashlib.sha256(CANONICAL_PATH.read_bytes()).hexdigest()
raw_news = load_raw_news(CANONICAL_PATH)
raw_snapshot = raw_news.copy(deep=True)
print(f"Raw DataFrame shape: {raw_news.shape}")
print(f"Canonical CSV SHA-256: {canonical_sha256_before}")
display(raw_news.head())
display(raw_news["sentiment"].value_counts().reindex(CLASS_ORDER, fill_value=0).rename("count").to_frame())

Raw DataFrame shape: (3453, 2)
Canonical CSV SHA-256: 791b78a0be391241bd6ea600fdd0128caee7c0dbe89eed10dfd00de505fb3c2e


,text,sentiment
0,"According to Gran , the company has no plans t...",neutral
1,With the new production plant the company woul...,positive
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,"In the third quarter of 2010 , net sales incre...",positive
4,Operating profit rose to EUR 13.1 mn from EUR ...,positive


,count
sentiment,
negative,420
neutral,2146
positive,887


### Policy `minimal-v1` (контракт для D2-02)

1. Декодировать HTML entities до стабильного результата.
2. Заменить корректные HTML tags/comments пробелами, сохранить текст элементов.
3. Применить `str.lower()`.
4. Свести Unicode whitespace к одному пробелу, убрать пробелы по краям.

Сохраняем отрицания, числа, проценты, валюты и пунктуацию. Не удаляем stopwords,
не выполняем lemmatization и не заменяем uppercase слова на тикеры.
`preprocess_text(str) -> str`: неверный тип вызывает `TypeError`, пустой результат
возвращает `""`. Ожидается идемпотентность; import и вызов не требуют NLTK/network.
Полный контракт и ограничения HTML описаны в SPEC §7.

Пример **ожидаемого**, ещё не исполненного преобразования:
`Profit is NOT down -5% &amp; costs £2.50.` →
`profit is not down -5% & costs £2.50.`

### Будущая схема и исключения

Canonical `(N, 2)` → processed `(M, 5)`, `M <= N`:

| Колонка | Значение |
| --- | --- |
| `source_row_id` | 0-based позиция в canonical CSV до фильтрации |
| `text` | Исходный текст |
| `sentiment` | Исходная строковая метка |
| `text_clean` | Непустой очищенный текст |
| `label_id` | `0=negative`, `1=neutral`, `2=positive` |

Неверная схема, тип или неизвестный класс — явные ошибки.
Missing text/label, whitespace-only и пустой результат очистки — исключения
с source IDs и причинами. Строка учитывается один раз, даже при нескольких причинах.
Исходные данные и порядок сохраняются. Дубликаты пока сохраняются, conflicts
потребуют явного решения; в Day03 совпадающие raw/clean texts группируются при split.
Эти поля и отчёты **ещё не создаются** на checkpoint D2-01.

In [3]:
pd.testing.assert_frame_equal(raw_news, raw_snapshot)
pd.testing.assert_frame_equal(raw_news, load_raw_news(CANONICAL_PATH))
canonical_sha256_after = hashlib.sha256(CANONICAL_PATH.read_bytes()).hexdigest()
assert canonical_sha256_before == canonical_sha256_after
assert tuple(raw_news.columns) == ("text", "sentiment")
print("Verified: DataFrame and canonical file unchanged; repeated load preserves rows/order.")
print("D2-01 complete: policy and raw-data setup; preprocessing implementation starts in D2-02.")

Verified: DataFrame and canonical file unchanged; repeated load preserves rows/order.
D2-01 complete: policy and raw-data setup; preprocessing implementation starts in D2-02.


### Интерпретация и ограничения

Вывод shape/counts относится к реальному локальному canonical CSV.
Повторная загрузка совпала с исходной таблицей; checksum файла не изменился.
Это не доказывает корректность будущей очистки: её проверки появятся в D2-02.
Числовые class IDs — категории, а не шкала интенсивности тональности.
Lowercase теряет регистр; минимальная HTML policy может принять `<USD>` за tag.
Рост macro F1 не обещаем: модель ещё не обучалась.

### Самопроверка

- Как удаление `not` и `-5%` может изменить смысл предложения?
- Почему `None` нельзя незаметно превращать в строку `"None"`?
- Зачем source ID назначается до фильтрации, а не после неё?
- Почему очистка не требует `fit`, а TF-IDF требует train?
- Что проверено на D2-01, а что остаётся проверить в D2-02?

### Повторный запуск

Выберите kernel **Python (Text-classification venv)**, затем Restart Kernel → Run All.
Из корня: `venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=text-classification notebooks/day02_preprocessing.ipynb`.
Из `notebooks/`: `../venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=text-classification day02_preprocessing.ipynb`.
Типы всех code cells: `venv/bin/python scripts/check_types.py` из корня.

## D2-02 — очистка одного текста

**Цель:** реализовать `preprocess_text(str) -> str` по policy `minimal-v1`.

**Краткая теория.** Здесь нет обучаемых параметров: одинаковый текст и policy
дают одинаковый результат без `fit`. Декодируем entities, заменяем tags/comments
пробелами, нормализуем регистр и whitespace. Повторяем до стабильного результата
для вложенного encoding и идемпотентности. Отрицания, суммы, проценты и пунктуация
сохраняются. NLTK resources и network не нужны.

**Вызов кода:** возьмём первое реальное предложение каждого класса в порядке
`negative`, `neutral`, `positive`. Source position показываем только для проверки
примеров, processed dataset и его source IDs появятся в D2-03.

In [4]:
from finnews_sentiment.features.preprocess import PREPROCESSING_POLICY, preprocess_text

example_rows: list[dict[str, str | int]] = []
for sentiment in CLASS_ORDER:
    source_position = next(
        position for position, value in enumerate(raw_news["sentiment"])
        if value == sentiment
    )
    raw_value = raw_news.iloc[source_position]["text"]
    if not isinstance(raw_value, str):
        raise TypeError("Real example text must be str")
    cleaned_value = preprocess_text(raw_value)
    assert preprocess_text(cleaned_value) == cleaned_value
    example_rows.append({
        "source_position": source_position,
        "sentiment": sentiment,
        "text": raw_value,
        "text_clean": cleaned_value,
    })

print(f"Preprocessing policy: {PREPROCESSING_POLICY}")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(example_rows))
for example in example_rows:
    print(f"\n{example['sentiment']} (source position {example['source_position']})")
    print(f"raw:   {example['text']}")
    print(f"clean: {example['text_clean']}")

pd.testing.assert_frame_equal(raw_news, raw_snapshot)
assert hashlib.sha256(CANONICAL_PATH.read_bytes()).hexdigest() == canonical_sha256_before
print("Verified: examples are idempotent; source DataFrame and CSV remain unchanged.")

Preprocessing policy: minimal-v1


,source_position,sentiment,text,text_clean
0,393,negative,"Jan. 6 -- Ford is struggling in the face of slowing truck and SUV sales and a surfeit of up-to-date , gotta-have cars .","jan. 6 -- ford is struggling in the face of slowing truck and suv sales and a surfeit of up-to-date , gotta-have cars ."
1,0,neutral,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .","according to gran , the company has no plans to move all production to russia , although that is where the company is growing ."
2,1,positive,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,with the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .



negative (source position 393)
raw:   Jan. 6 -- Ford is struggling in the face of slowing truck and SUV sales and a surfeit of up-to-date , gotta-have cars .
clean: jan. 6 -- ford is struggling in the face of slowing truck and suv sales and a surfeit of up-to-date , gotta-have cars .

neutral (source position 0)
raw:   According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .
clean: according to gran , the company has no plans to move all production to russia , although that is where the company is growing .

positive (source position 1)
raw:   With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .
clean: with the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefo

### Интерпретация

В реальных примерах регистр нормализуется, внешние и повторные пробелы убираются
при наличии. Числа, десятичные точки, `%`, скобки и слова сохраняются.
В neutral-примере сохранено отрицание `no plans`; в negative-примере
сохранены `Jan. 6`, `--` и дефисы, а `SUV` становится `suv`. Метки не меняются.
Эти три примера демонстрируют lowercase; валюты и проценты покрыты offline tests.
Если в этих предложениях нет HTML, они не подтверждают HTML boundary cases:
entities, tags, comments, сравнения и неверные типы проверяются отдельными offline tests.
Идемпотентность трёх примеров и сохранность исходника подтверждены assertions.

**Ограничения:** lowercase теряет регистр, `<USD>` может считаться tag;
это cleanup предложений, не HTML scraping. Нет измерения macro F1.
Табличные helpers в модуле остаются заготовками: текущий checkpoint вызывает
только `preprocess_text`. Исключения строк, mapping и duplicate policy — следующие шаги.

### Самопроверка

- Почему здесь нет `fit`, vocabulary и обучаемых параметров?
- Что произойдёт с `None`, пустой строкой и строкой из HTML tags?
- Почему удаление цифр, валют или `not` сейчас не входит в policy?
- Зачем проверять `preprocess_text(preprocess_text(s)) == preprocess_text(s)`?
- Почему три реальных примера не заменяют tests на HTML и неверные типы?

## D2-03 — обработка таблицы и отчёт об исключениях

**Цель:** получить новую таблицу с `source_row_id`, исходным текстом/меткой
и `text_clean`, не меняя canonical данные.

**Краткая теория.** `preprocess_text` преобразует одну строку. `preprocess_news`
проверяет схему всей таблицы и отделяет ошибки контракта от допустимых missing/blank
примеров. Source IDs — позиции в исходном CSV до фильтрации, а не pandas index.
Повторные тексты сохраняются; классы остаются строками.

Причины исключения: `missing_text`, `missing_sentiment`, `blank_text`,
`empty_after_cleaning`. Одна строка может иметь несколько причин, но учитывается
один раз: `N_input = N_retained + N_excluded`. Ошибочный класс/тип не маскируется
удалением. Если исключены все строки, exception содержит report.

**Вызов кода:** обработаем весь реальный canonical dataset в памяти.
Processed CSV и reports пока не записываем — это D2-05.

In [ ]:
from dataclasses import asdict
from finnews_sentiment.features.preprocess import preprocess_news

processed_news, preprocessing_report = preprocess_news(raw_news)
assert preprocessing_report.n_input == preprocessing_report.n_retained + preprocessing_report.n_excluded
assert tuple(processed_news.columns) == ("source_row_id", "text", "sentiment", "text_clean")
assert processed_news["source_row_id"].is_unique
assert processed_news["source_row_id"].is_monotonic_increasing
assert processed_news["text_clean"].map(bool).all()
pd.testing.assert_frame_equal(raw_news, raw_snapshot)
assert hashlib.sha256(CANONICAL_PATH.read_bytes()).hexdigest() == canonical_sha256_before

print(f"Raw shape: {raw_news.shape}; processed shape: {processed_news.shape}")
print(f"Input={preprocessing_report.n_input}, retained={preprocessing_report.n_retained}, excluded={preprocessing_report.n_excluded}")
print(f"Reason counts: {preprocessing_report.reason_counts}")
print(f"Missing labels before: {preprocessing_report.missing_sentiment_before}")
print(f"Changed retained texts: {len(preprocessing_report.changed_source_row_ids)}")
display(pd.DataFrame({
    "before": preprocessing_report.class_counts_before,
    "after": preprocessing_report.class_counts_after,
}).reindex(CLASS_ORDER))
excluded_details = [asdict(row) for row in preprocessing_report.excluded_rows]
print(f"Excluded source rows and reasons: {excluded_details}")
with pd.option_context("display.max_colwidth", None):
    display(processed_news.head(3))

Raw shape: (3453, 2); processed shape: (3453, 4)
Input=3453, retained=3453, excluded=0
Reason counts: {'missing_text': 0, 'missing_sentiment': 0, 'blank_text': 0, 'empty_after_cleaning': 0}
Missing labels before: 0
Changed retained texts: 3447


,before,after
negative,420,420
neutral,2146,2146
positive,887,887


Excluded source rows and reasons: []


,source_row_id,text,sentiment,text_clean
0,0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .",neutral,"according to gran , the company has no plans to move all production to russia , although that is where the company is growing ."
1,1,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,positive,with the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .
2,2,"For the last quarter of 2010 , Componenta 's net sales doubled to EUR131m from EUR76m for the same period a year earlier , while it moved to a zero pre-tax profit from a pre-tax loss of EUR7m .",positive,"for the last quarter of 2010 , componenta 's net sales doubled to eur131m from eur76m for the same period a year earlier , while it moved to a zero pre-tax profit from a pre-tax loss of eur7m ."


### Интерпретация

Фактический результат: `(3453, 2)` → `(3453, 4)`; сохранены все 3453 строки,
исключено 0. Все четыре reason counts равны 0; классы до/после: negative=420,
neutral=2146, positive=887. Изменено представление 3447 сохранённых текстов.
Shape и counts выше относятся к реальному Financial PhraseBank. Нулевые исключения
означают, что ни одна строка не нарушила критерии пригодности этой policy;
это не доказывает отсутствие ошибок разметки. `changed retained texts` считает
строки с изменённым представлением, а не изменёнными метками.
Counts классов до/после позволяют увидеть влияние исключений на распределение.

В processed таблице четыре колонки; `label_id` появится в D2-04.
Исходный текст сохранён рядом с очищенным. Source IDs позволяют сопоставить
результат с исходной строкой. Дубликаты не удалялись, splits ещё не создавались.
Проверка missing/blank и пересекающихся причин выполнена на fixtures в tests,
которые не подменяют реальные данные этого notebook.

### Самопроверка

- Чем missing value отличается от буквальной строки `"nan"`?
- Почему сумма counts отдельных причин может превышать число исключённых строк?
- Почему ошибка неизвестного класса останавливает обработку всей таблицы?
- Зачем сохранять source IDs с пропусками после фильтрации?
- Что показывают нулевые исключения, а чего они не подтверждают?

## D2-04 — class IDs и duplicate/conflict policy

**Цель:** добавить `label_id` и исследовать совпадения raw/clean texts.

**Теория:** фиксированное `negative=0`, `neutral=1`, `positive=2` задаёт категории,
а не числовую интенсивность. Mapping не обучается и не зависит от наличия класса.
Повторы с одной меткой сохраняем. Несколько меток для одного raw/clean text —
conflict: диагностика доступна, но финальная подготовка блокируется.

Нормализация может объединить разные raw texts. Совпадающие предложения между
train/validation/test создают риск leakage: в Day03 вся группа должна попасть
в один split. Сейчас splits не создаются и дубликаты не удаляются.

**Вызов кода:** сначала диагностируем реальные данные, затем добавим IDs через
API, которое блокирует conflicts. Все результаты остаются в памяти.

In [6]:
from finnews_sentiment.features.labels_duplicates import (
    LABEL_MAPPING, analyze_duplicates, prepare_labeled_news,
)

duplicate_report = analyze_duplicates(processed_news)
print(f"Label mapping: {LABEL_MAPPING}")
print(f"Raw duplicate groups: {len(duplicate_report.raw_groups)}; rows beyond first: {duplicate_report.raw_duplicate_rows}")
print(f"Clean duplicate groups: {len(duplicate_report.clean_groups)}; rows beyond first: {duplicate_report.clean_duplicate_rows}")
print(f"New clean groups: {sum(group.new_after_cleaning for group in duplicate_report.clean_groups)}")
print(f"Raw conflicts: {sum(group.conflicting for group in duplicate_report.raw_groups)}")
print(f"Clean conflicts: {sum(group.conflicting for group in duplicate_report.clean_groups)}")
for representation, groups in (("raw", duplicate_report.raw_groups), ("clean", duplicate_report.clean_groups)):
    print(f"\n{representation} groups (all source IDs):")
    for group in groups:
        print(f"IDs={group.source_row_ids}, labels={group.sentiments}, new={group.new_after_cleaning}, conflict={group.conflicting}")
        print(group.text)

labeled_news, checked_duplicate_report = prepare_labeled_news(processed_news)
assert checked_duplicate_report == duplicate_report
assert len(labeled_news) == len(processed_news)
assert labeled_news["source_row_id"].tolist() == processed_news["source_row_id"].tolist()
pd.testing.assert_frame_equal(labeled_news.drop(columns="label_id"), processed_news)
pd.testing.assert_frame_equal(raw_news, raw_snapshot)
assert hashlib.sha256(CANONICAL_PATH.read_bytes()).hexdigest() == canonical_sha256_before
print(f"\nLabeled shape: {labeled_news.shape}")
display(labeled_news.groupby(["sentiment", "label_id"]).size().rename("count").to_frame())
display(labeled_news.head())

Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
Raw duplicate groups: 5; rows beyond first: 5
Clean duplicate groups: 5; rows beyond first: 5
New clean groups: 0
Raw conflicts: 0
Clean conflicts: 0

raw groups (all source IDs):
IDs=(773, 774), labels=('neutral', 'neutral'), new=False, conflict=False
The issuer is solely responsible for the content of this announcement .
IDs=(958, 959), labels=('neutral', 'neutral'), new=False, conflict=False
The report profiles 614 companies including many key and niche players worldwide such as Black & Decker Corporation , Fiskars Corporation , Fiskars Brands , Inc. , Husqvarna Outdoor Products Inc. , K+S Group , Ryobi Technologies , Inc. , The Scotts Miracle-Gro Company , and Van Group , Inc. .
IDs=(1544, 1545), labels=('neutral', 'neutral'), new=False, conflict=False
Ahlstrom 's share is quoted on the NASDAQ OMX Helsinki .
IDs=(1681, 1682), labels=('neutral', 'neutral'), new=False, conflict=False
SSH Communications Security Corporation i

,,count
sentiment,label_id,
negative,0,420
neutral,1,2146
positive,2,887


,source_row_id,text,sentiment,text_clean,label_id
0,0,"According to Gran , the company has no plans t...",neutral,"according to gran , the company has no plans t...",1
1,1,With the new production plant the company woul...,positive,with the new production plant the company woul...,2
2,2,"For the last quarter of 2010 , Componenta 's n...",positive,"for the last quarter of 2010 , componenta 's n...",2
3,3,"In the third quarter of 2010 , net sales incre...",positive,"in the third quarter of 2010 , net sales incre...",2
4,4,Operating profit rose to EUR 13.1 mn from EUR ...,positive,operating profit rose to eur 13.1 mn from eur ...,2


### Интерпретация

Фактический результат: `(3453, 4)` → `(3453, 5)`. Raw и clean: по 5 групп,
по 5 повторов сверх первого. Новых clean groups=0, conflicts=0. Все 3453 строки
сохранены; mapping: negative=0 (420), neutral=1 (2146), positive=2 (887).

Counts выше измерены на реальном dataset; группы показывают исходные позиции,
метки и точный ключ совпадения. `rows beyond first` — число повторов сверх первой
строки группы, а не количество всех участников. Raw и clean counts пересекаются.
`new=True` означает слияние разных raw texts после очистки, а не обязательно
отсутствие raw duplicates внутри группы.

Успешный `prepare_labeled_news` подтверждает отсутствие conflicts по этой policy.
Он не подтверждает правильность разметки всех предложений и не ищет парафразы.
Все строки сохраняются; повторы могут увеличивать вес соответствующих примеров.
Если появится conflict, исключение сохранит `.report` для явного решения;
автоматического majority vote нет. Processed CSV пока не записан.

### Самопроверка

- Означает ли `positive=2`, что он вдвое больше `neutral=1`?
- Почему mapping должен сохраняться, если в небольшом наборе нет negative?
- Чем raw duplicate отличается от нового совпадения после lowercase?
- Почему нельзя просто выбрать первую метку конфликтного текста?
- Почему сохранённые дубликаты требуют группового split в Day03?

## D2-05 — offline подготовка и provenance

**Цель:** сохранить processed CSV и три JSON reports, подтвердить повторяемость.

**Теория:** CSV хранит значения, но не объясняет источник, policy или class order.
Provenance связывает результат с pinned archive/revision, canonical checksum,
версиями policy/runtime и label mapping. Source IDs и порядок сохраняются.
Команда сверяет canonical данные с локальным архивом и блокирует conflicts
до записи. Здесь нет download, split или обучения.

**Явное действие:** следующая ячейка вызывает общий `prepare_day02(ROOT)`
и перезаписывает четыре локальных производных файла. Raw/interim неизменны.
Два последовательных запуска проверят одинаковые bytes.

Offline команда из корня:
`PYTHONPATH=src venv/bin/python -m finnews_sentiment.data.prepare_day02`.
Из `notebooks/`:
`PYTHONPATH=../src ../venv/bin/python -m finnews_sentiment.data.prepare_day02`.
Инструкции и ограничения сохранены в `docs/day02.md`.

In [7]:
import json
from finnews_sentiment.data.prepare_day02 import prepare_day02

archive_path_day02 = ROOT / "data/raw/FinancialPhraseBank-v1.0.zip"
archive_hash_before_day02 = hashlib.sha256(archive_path_day02.read_bytes()).hexdigest()
day02_artifacts = prepare_day02(ROOT)
artifact_paths = (
    day02_artifacts.processed_csv, day02_artifacts.preprocessing_report,
    day02_artifacts.duplicate_report, day02_artifacts.provenance,
)
artifact_bytes_first = [path.read_bytes() for path in artifact_paths]
prepare_day02(ROOT)
assert artifact_bytes_first == [path.read_bytes() for path in artifact_paths]

saved_news = pd.read_csv(
    day02_artifacts.processed_csv, encoding="utf-8", keep_default_na=False,
    dtype={"source_row_id": "int64", "text": "str", "sentiment": "str", "text_clean": "str", "label_id": "int64"},
)
pd.testing.assert_frame_equal(saved_news, labeled_news)
pd.testing.assert_frame_equal(raw_news, raw_snapshot)
assert hashlib.sha256(CANONICAL_PATH.read_bytes()).hexdigest() == canonical_sha256_before
assert hashlib.sha256(archive_path_day02.read_bytes()).hexdigest() == archive_hash_before_day02
saved_provenance = json.loads(day02_artifacts.provenance.read_text(encoding="utf-8"))
saved_preprocessing = json.loads(day02_artifacts.preprocessing_report.read_text(encoding="utf-8"))
saved_duplicates = json.loads(day02_artifacts.duplicate_report.read_text(encoding="utf-8"))
assert saved_provenance["output"]["sha256"] == hashlib.sha256(day02_artifacts.processed_csv.read_bytes()).hexdigest()
print(f"Saved CSV shape: {saved_news.shape}")
for path in artifact_paths:
    print(f"{path.relative_to(ROOT)}: {path.stat().st_size} bytes")
print(f"Retained={saved_preprocessing['n_retained']}; excluded={saved_preprocessing['n_excluded']}")
print(f"Raw/clean repeats: {saved_duplicates['raw_duplicate_rows']}/{saved_duplicates['clean_duplicate_rows']}")
print(f"Versions: {saved_provenance['versions']}")
print(f"Input SHA-256: {saved_provenance['input']['sha256']}")
print(f"Output SHA-256: {saved_provenance['output']['sha256']}")
print("Verified: CSV roundtrip, identical output bytes on repeated offline run, immutable raw/interim.")
display(saved_news.head())

Saved CSV shape: (3453, 5)
data/processed/financial_phrasebank_75agree.csv: 929290 bytes
reports/day02/preprocessing.json: 33854 bytes
reports/day02/duplicates.json: 3710 bytes
reports/day02/provenance.json: 1800 bytes
Retained=3453; excluded=0
Raw/clean repeats: 5/5
Versions: {'python': '3.14.7', 'pandas': '3.0.6', 'numpy': '2.5.3'}
Input SHA-256: 791b78a0be391241bd6ea600fdd0128caee7c0dbe89eed10dfd00de505fb3c2e
Output SHA-256: 39dda898a281dc3294be081917b0681524e73edb60f00913183c67714da0c448
Verified: CSV roundtrip, identical output bytes on repeated offline run, immutable raw/interim.


,source_row_id,text,sentiment,text_clean,label_id
0,0,"According to Gran , the company has no plans t...",neutral,"according to gran , the company has no plans t...",1
1,1,With the new production plant the company woul...,positive,with the new production plant the company woul...,2
2,2,"For the last quarter of 2010 , Componenta 's n...",positive,"for the last quarter of 2010 , componenta 's n...",2
3,3,"In the third quarter of 2010 , net sales incre...",positive,"in the third quarter of 2010 , net sales incre...",2
4,4,Operating profit rose to EUR 13.1 mn from EUR ...,positive,operating profit rose to eur 13.1 mn from eur ...,2


### Интерпретация и ограничения

Вывод выше подтверждает сохранение реальных данных и воспроизводимость четырёх
outputs при текущих входах/runtime. CSV содержит пять согласованных колонок;
исключения и группы дубликатов описаны в JSON reports. Метки, source IDs и порядок
после roundtrip совпали с результатом D2-04. Изменение policy/runtime требует
нового запуска; checksum CSV связывает provenance с конкретными bytes.

Артефакты исключены из Git; общий код и команда позволяют воспроизвести их.
Локальный архив обязателен для сверки canonical CSV. Запись четырёх файлов
не является единой транзакцией; после I/O failure может потребоваться повторный запуск.
Качество классификации не измерено. Итоговые выводы Day02 — D2-06.

### Самопроверка

- Почему одного processed CSV недостаточно для воспроизводимости?
- Какие checksums подтверждают связь результата с исходными данными?
- Почему numeric-only text и строка `NA` требуют явных параметров чтения CSV?
- Что произойдёт при конфликтных метках или изменённом canonical CSV?
- Почему в воспроизводимых reports нет timestamp текущего запуска?

## D2-06 — ручная проверка и итоговая приёмка

**Цель:** проверить результат всего Day02 и сохранить фактические выводы.

**Краткая теория.** Детерминированный preprocessing не имеет `fit` и не изучает
статистику корпуса. Наличие стабильного результата и корректной схемы необходимо
для ML pipeline, но не доказывает рост macro F1. Числовые IDs кодируют категории.
Сохранённые повторные тексты потребуют группового split в Day03; TF-IDF vocabulary
и IDF будут fit только на train. Ни split, ни обучение сейчас не выполняются.

**Вызов кода:** первые три реальных предложения каждого класса в порядке
источника. Это ручной sanity check, не случайная репрезентативная оценка всей policy.
Полные raw → clean пары показываем вместе с `sentiment`, `label_id` и source IDs.

In [8]:
review_examples = pd.concat(
    [saved_news.loc[saved_news["sentiment"] == sentiment].head(3) for sentiment in CLASS_ORDER],
    ignore_index=True,
)
assert len(review_examples) == 9
with pd.option_context("display.max_colwidth", None):
    display(review_examples[["source_row_id", "text", "text_clean", "sentiment", "label_id"]])
for example in review_examples.itertuples(index=False):
    print(f"\nID={example.source_row_id}, sentiment={example.sentiment}, label_id={example.label_id}")
    print(f"raw:   {example.text}")
    print(f"clean: {example.text_clean}")

assert saved_news["text"].map(preprocess_text).equals(saved_news["text_clean"])
assert saved_news["text_clean"].map(preprocess_text).equals(saved_news["text_clean"])
assert saved_preprocessing["n_input"] == saved_preprocessing["n_retained"] + saved_preprocessing["n_excluded"]
assert saved_news["sentiment"].map(LABEL_MAPPING).tolist() == saved_news["label_id"].tolist()
assert not analyze_duplicates(saved_news).has_conflicts
pd.testing.assert_frame_equal(raw_news, raw_snapshot)
print("Verified across all retained rows: cleanup consistency, idempotence and fixed labels.")

,source_row_id,text,text_clean,sentiment,label_id
0,393,"Jan. 6 -- Ford is struggling in the face of slowing truck and SUV sales and a surfeit of up-to-date , gotta-have cars .","jan. 6 -- ford is struggling in the face of slowing truck and suv sales and a surfeit of up-to-date , gotta-have cars .",negative,0
1,396,Pharmaceuticals group Orion Corp reported a fall in its third-quarter earnings that were hit by larger expenditures on R&D and marketing .,pharmaceuticals group orion corp reported a fall in its third-quarter earnings that were hit by larger expenditures on r&d and marketing .,negative,0
2,397,"However , the growth margin slowed down due to the financial crisis .","however , the growth margin slowed down due to the financial crisis .",negative,0
3,0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .","according to gran , the company has no plans to move all production to russia , although that is where the company is growing .",neutral,1
4,48,"At the request of Finnish media company Alma Media 's newspapers , research manager Jari Kaivo-oja at the Finland Futures Research Centre at the Turku School of Economics has drawn up a future scenario for Finland 's national economy by using a model developed by the University of Denver .","at the request of finnish media company alma media 's newspapers , research manager jari kaivo-oja at the finland futures research centre at the turku school of economics has drawn up a future scenario for finland 's national economy by using a model developed by the university of denver .",neutral,1
5,49,"In Sweden , Gallerix accumulated SEK denominated sales were down 1 % and EUR denominated sales were up 11 % .","in sweden , gallerix accumulated sek denominated sales were down 1 % and eur denominated sales were up 11 % .",neutral,1
6,1,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,with the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,positive,2
7,2,"For the last quarter of 2010 , Componenta 's net sales doubled to EUR131m from EUR76m for the same period a year earlier , while it moved to a zero pre-tax profit from a pre-tax loss of EUR7m .","for the last quarter of 2010 , componenta 's net sales doubled to eur131m from eur76m for the same period a year earlier , while it moved to a zero pre-tax profit from a pre-tax loss of eur7m .",positive,2
8,3,"In the third quarter of 2010 , net sales increased by 5.2 % to EUR 205.5 mn , and operating profit by 34.9 % to EUR 23.5 mn .","in the third quarter of 2010 , net sales increased by 5.2 % to eur 205.5 mn , and operating profit by 34.9 % to eur 23.5 mn .",positive,2



ID=393, sentiment=negative, label_id=0
raw:   Jan. 6 -- Ford is struggling in the face of slowing truck and SUV sales and a surfeit of up-to-date , gotta-have cars .
clean: jan. 6 -- ford is struggling in the face of slowing truck and suv sales and a surfeit of up-to-date , gotta-have cars .

ID=396, sentiment=negative, label_id=0
raw:   Pharmaceuticals group Orion Corp reported a fall in its third-quarter earnings that were hit by larger expenditures on R&D and marketing .
clean: pharmaceuticals group orion corp reported a fall in its third-quarter earnings that were hit by larger expenditures on r&d and marketing .

ID=397, sentiment=negative, label_id=0
raw:   However , the growth margin slowed down due to the financial crisis .
clean: however , the growth margin slowed down due to the financial crisis .

ID=0, sentiment=neutral, label_id=1
raw:   According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .
clean: a

### Ручная интерпретация девяти примеров

- **Negative, IDs 393/396/397:** сохранены число `6`, `--`, дефисы, `R&D` →
  `r&d`, слова `fall`, `slowed down` и `financial crisis`. Регистр нормализован;
  негативные формулировки не удалены.
- **Neutral, IDs 0/48/49:** `no plans` сохраняет отрицание; апострофы и дефисы
  в именах остаются. В ID 49 сохранены `SEK`/`EUR` → `sek`/`eur`, `1 %`, `11 %`
  и обе формулировки `down`/`up`. Их наличие само по себе не переопределяет метку.
- **Positive, IDs 1/2/3:** сохранены `increase`, `improve`, `doubled`,
  `EUR131m`, `EUR76m`, `EUR7m` (в lowercase), `5.2 %`, `34.9 %`, `205.5`,
  `23.5`, `pre-tax` и `loss`. Слово `loss` встречается в положительном предложении:
  тональность определяется контекстом, а не одной keyword.

В этих девяти парах не обнаружена потеря перечисленных сигналов. Это ограниченная
ручная проверка; HTML/entities и неверные типы подтверждаются offline tests.
Assertions выше отдельно проверяют все сохранённые строки на согласованность
с policy и идемпотентность, но не доказывают семантическую корректность каждого текста.

### Итоги и ограничения Day02

Все 3453 строки сохранены, исключений 0; форма `(3453, 2)` → `(3453, 5)`.
Изменено представление 3447 текстов, class counts и метки сохранены.
Raw/clean имеют по 5 duplicate groups и 5 repeats сверх первого;
новых clean groups и conflicts нет. Policy не ищет парафразы или ошибочную разметку.
Lowercase теряет регистр, `<USD>` может восприниматься как tag; это не web scraping.
Сохранение повторов влияет на вес примеров; Day03 должен соблюдать group boundary.
Модель не обучена, качество классификации и изменение macro F1 не измерены.

Следующая ячейка сохраняет narrative в локальный `reports/day02/observations.md`.
Отчёт пересоздаётся из проверенных результатов notebook; проверки pytest/type gate
выполняются отдельными командами и фиксируются в PLAN.

In [9]:
observations_path = ROOT / "reports/day02/observations.md"
observations_text = f"""# Day02 — наблюдения и ограничения

Источник: {saved_provenance['source']['dataset']}, subset={saved_provenance['source']['subset']}.
Revision: {saved_provenance['source']['revision']}.
Policy: {saved_provenance['preprocessing_policy']}.
Runtime: {saved_provenance['versions']}.

## Фактический результат

- Input={saved_preprocessing['n_input']}; retained={saved_preprocessing['n_retained']}; excluded={saved_preprocessing['n_excluded']}.
- Processed shape: {saved_news.shape}; колонки: {list(saved_news.columns)}.
- Причины исключения: {saved_preprocessing['reason_counts']}.
- Изменено представление {len(saved_preprocessing['changed_source_row_ids'])} сохранённых текстов.
- Class counts до: {saved_preprocessing['class_counts_before']}; после: {saved_preprocessing['class_counts_after']}.
- Mapping: {saved_provenance['label_mapping']}.
- Raw/clean duplicate groups: {len(saved_duplicates['raw_groups'])}/{len(saved_duplicates['clean_groups'])}; repeats сверх первого: {saved_duplicates['raw_duplicate_rows']}/{saved_duplicates['clean_duplicate_rows']}.
- Новых clean groups={saved_duplicates['new_clean_groups']}; raw/clean conflicts={saved_duplicates['raw_conflicts']}/{saved_duplicates['clean_conflicts']}.
- Input SHA-256: {saved_provenance['input']['sha256']}.
- Output SHA-256: {saved_provenance['output']['sha256']}.

## Ручная проверка

Первые три предложения каждого класса; source IDs: {review_examples['source_row_id'].tolist()}.
Negative: сохранены 6, --, дефисы, R&D и fall/slowed down/financial crisis.
Neutral: сохранены no plans, апострофы/дефисы, SEK/EUR, 1 % и 11 %, down/up.
Positive: сохранены increase/improve/doubled, EUR131m/EUR76m/EUR7m,
5.2 %, 34.9 %, 205.5, 23.5, pre-tax и loss; регистр нормализован.
Перечисленные сигналы в девяти парах не потеряны; метки не пересматривались.
Полные raw/clean пары сохранены в notebook.

## Проверки и ограничения

Notebook assertions подтвердили roundtrip, одинаковые bytes четырёх outputs
при повторной offline подготовке, неизменность raw/interim, согласованность
cleanup и идемпотентность всех retained texts. Offline tests, type gate и
clean-kernel execution подтверждаются отдельно в PLAN.md.
Ручная выборка не репрезентативна и не доказывает смысловую корректность всего корпуса.
Нулевые исключения не доказывают правильность разметки. Совпадения ищутся точно,
без fuzzy matching. Lowercase теряет регистр; <USD> может восприниматься как tag.
Повторы сохранены и могут менять веса примеров; в Day03 нужен групповой split.
Нет fit/TF-IDF/splits/training, macro F1 и его рост не измерены.
Запись файлов не является единой транзакцией; I/O failure требует повторного запуска.

## Воспроизведение

Из корня: PYTHONPATH=src venv/bin/python -m finnews_sentiment.data.prepare_day02
Затем Restart Kernel / Run All для notebooks/day02_preprocessing.ipynb.
CLI сохраняет CSV и три JSON; этот narrative создаётся разделом D2-06 notebook.
"""
observations_path.write_text(observations_text, encoding="utf-8")
print(f"Saved: {observations_path.relative_to(ROOT)}")
print("Day02 data contracts confirmed; training/evaluation remains future work.")

Saved: reports/day02/observations.md
Day02 data contracts confirmed; training/evaluation remains future work.


### Самопроверка

- Какие преобразования Day02 не обучаются, а что потребует `fit` в Day03?
- Почему `loss` не всегда означает negative?
- Что доказала ручная проверка девяти примеров и чего она не доказала?
- Почему одинаковые raw/clean texts должны попасть в один split?
- Почему нельзя обещать рост macro F1 по результатам preprocessing?

Следующий этап: D3-01 — отдельная детализация и реализация group-aware splits.
Day03 на этом checkpoint не начат.